**딥러닝(Deep Learning: DL)** 은 다수 은닉층(hidden layer)을 가진 인공신경망을 적용한 기법

딥러닝의 기본 엔진으로는 Tensorflow을 주로 사용. 간편한 인터페이스를 위한 엔진으로는 Keras가 존재.

# 6.1. MF를 신경망으로 변환하기

먼저 은닉층이 없는 신경망 모형을 Keras로 구현해볼 예정. 은닉층이 없는 신경망 모델은 MF 알고리즘과 기본적으로는 같은 모형.

![딥러닝](../static/img_6.png)

## One-Hot Representation

### One-Hot Representation이란?

One-Hot Representation은 여러 개의 범주(category) 중 딱 하나만 1이고 나머지는 전부 0인 벡터 표현.

예를 들어,
	•	사용자 수 = 5명
	•	user_id = 2번

이면 벡터는: $\text{user}_2 = [0, 1, 0, 0, 0]$

### 추천 시스템에서 왜 One-Hot이 등장하는 이유

신경망은 숫자 벡터만 입력으로 받을 수 있기 때문.
- user_id = 2
- item_id = 10

이런 정수 ID 자체에는 의미가 없음 (2가 1보다 두 배 더 중요한 게 아니기에)

그래서 ID → 벡터로 바꿔주는 과정이 필요

→ 그 가장 원초적인 방법이 One-Hot Encoding

### One-Hot Representation Layer의 역할

추천 모델 입력 흐름

```
user_id ─▶ One-Hot Vector
item_id ─▶ One-Hot Vector
```

이 단계에서:
- 사용자 ID → 사용자 공간의 좌표
- 아이템 ID → 아이템 공간의 좌표


## MF to Deep Learning

### 기존 Matrix Factorization(MF) 요약

MF: 사용자 벡터와 아이템 벡터를 곱해서 평점을 예측한다

수식으로는 아래와 같음.

$$
\hat r_{ui} = \mathbf{p}_u^\top \mathbf{q}_i
$$
$$
\mathbf{p}_u: 사용자 임베딩 벡터, \quad
\mathbf{q}_i: 아이템 임베딩 벡터
$$

### 이걸 “신경망 관점”으로 보면?

여기서 관점을 살짝만 바꿔보자.

👉 사용자 ID, 아이템 ID는 사실상 입력값
- 입력: (user_id, item_id)
- 출력: 예측 평점

그럼 중간에서 무슨 일이 일어나지?

👉 Embedding lookup = 신경망의 첫 번째 레이어
	•	user_id → 사용자 embedding vector
	•	item_id → 아이템 embedding vector

이건 딥러닝에서 말하는 Embedding Layer랑 완전히 같다.

### MF는 사실 이런 “초간단 신경망”이다

<pre>
user_id ──▶ [User Embedding] ──┐
                               ├─ dot product ─▶ predicted rating
item_id ──▶ [Item Embedding] ──┘
</pre>


🔑 여기서 중요한 포인트:
- Embedding = 학습 가능한 파라미터 (가중치)
- dot product = 고정된 연산 (비선형 없음)
- hidden layer 없음

👉 그래서 MF는
“Hidden layer가 없는 신경망” 또는 “선형 모델” 로 볼 수 있다.

### Deep Learning으로 바꾼다는 건, 뭘 바꾸는 걸까?

**MF의 한계**
- 사용자와 아이템 관계가 선형(linear) 이라고 가정
- “A와 B가 동시에 좋을 때만 점수가 튄다” 같은 복잡한 패턴 표현 불가

→ hookup 점수(내적) 말고 신경망이 직접 관계를 학습하게 하자.

### MF → Neural Network 로의 핵심 변화

(1) dot product를 버린다 (또는 약화시킨다)

기존:
$\hat r_{ui} = \mathbf{p}_u^\top \mathbf{q}_i$

변경:
1. 사용자 벡터 + 아이템 벡터를 붙이고(concat)
2. MLP(다층 신경망)에 넣는다

$\hat r_{ui} = f_{\text{NN}}([\mathbf{p}_u ; \mathbf{q}_i])$

⸻

(2) 구조적으로 보면 이렇게 바뀐다

![MF_to_network](../static/img_7.png)

<pre>
user_id ─▶ Embedding ─┐  
                      ├─ concat ─▶ Dense ─▶ Dense ─▶ rating
item_id ─▶ Embedding ─┘  
</pre>

이제는:
- 관계가 비선형
- 복잡한 interaction 학습 가능
- feature 조합 자동 학습

